# 🔍 Phase 07 — XAI & Information Dynamics (cross-symbol, cross-layer, cross-horizon)

This notebook does **not** retrain the full model roster either. Questions Q1-Q3 below are
pure aggregation of the MI/SHAP ranking tables Phase 05 already computed and saved (train-only,
leakage-safe, exactly as scored there). Questions Q4-Q5 need SHAP computed under conditions Phase
05 never ran (regime-conditional, fold-conditional), so those two sections retrain **one small,
fixed, default-hyperparameter model** (never the full roster) purely to get a SHAP explainer under
each condition.

**Sections** (matching the review's recommended `Phase07_XAI_and_Information_Dynamics.ipynb`
outline):

- **Q1. Which features matter?** — feature-level SHAP, aggregated across symbols/layers/horizons.
- **Q2. Which information domains matter?** — Market / Macro / GPR / Sentiment / Sustainability /
  Cross-market / Regime, aggregated SHAP by bucket.
- **Q3. Does importance change with horizon?** — the same domain buckets across h=1,3,5,10,21.
- **Q4. Does importance change with regime?** — **new**: SHAP recomputed separately for Bull /
  Bear / Sideways TRAIN+VAL rows (regime label joined in from Phase 05 L1's `target_tri_{h}d`).
- **Q5. Is importance stable?** — **new**: SHAP recomputed per Purged Walk-Forward CV fold;
  reports each feature's rank-stability across folds.

This directly supports **H4** (does feature/domain importance shift with horizon) and **H5**
(does it shift with regime), and the report's own "external-derived features become more
prominent at longer horizons" observation is turned into an actual cross-horizon SHAP table here
rather than staying a narrative claim.


## 1 · Install & Imports

In [ ]:
!pip install -q lightgbm shap scikit-learn pandas numpy scipy pyarrow fastparquet tqdm


In [ ]:
import warnings, os, glob, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, norm as scipy_norm

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, matthews_corrcoef, roc_auc_score, log_loss, mean_squared_error,
    mean_absolute_error, r2_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import TimeSeriesSplit

try:
    import lightgbm as lgb
except ImportError:
    get_ipython().system('pip install -q lightgbm')
    import lightgbm as lgb

try:
    import shap
except ImportError:
    get_ipython().system('pip install -q shap')
    import shap

from IPython.display import display

print('✅ Imports ready.')


## 2 · Config

In [ ]:
# ── ⭐ Grid this notebook aggregates / sweeps over ⭐ ────────────────────────────────────────
# Must match Phase 05's own SYMBOL / HORIZON choices exactly -- this notebook reads the SAME
# Artifacts/ tree those three notebooks wrote to, keyed by (symbol, layer, horizon).
ALL_SYMBOLS = ['NIFTY50', 'BANKNIFTY', 'NIFTYNEXT50']
ALL_HORIZONS_FOR_STATS = [1, 3, 5, 10, 21]

# The single symbol/horizon this run's NEW computations (Sections 6/8 in Phase06, Q4/Q5 in
# Phase07) retrain against -- exactly the "two things you change" pattern from Phase 05. The
# pure-aggregation sections above/below read the FULL grid (ALL_SYMBOLS × ALL_HORIZONS_FOR_STATS)
# regardless of this setting.
SYMBOL  = 'NIFTYNEXT50'
HORIZON = 3

RANDOM_STATE = 42

# ── Paths -- IDENTICAL to Phase05_Unified_L1_v4 / L2_v4 / L3_v4's own Section 2 ─────────────
GDRIVE_BASE = '/content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107'
DATA_ROOT   = os.path.join(GDRIVE_BASE, 'Data')
FEAT_DIR    = os.path.join(DATA_ROOT, 'features_v6')
FS_DIR      = os.path.join(DATA_ROOT, 'feature_selection_v6')

ARTIFACT_ROOT     = os.path.join(GDRIVE_BASE, 'Artifacts')
MODEL_SAVE_DIR    = os.path.join(ARTIFACT_ROOT, 'models')
PRED_SAVE_DIR     = os.path.join(ARTIFACT_ROOT, 'predictions')
FEATIMP_SAVE_DIR  = os.path.join(ARTIFACT_ROOT, 'feature_importance')
RESULTS_SAVE_DIR  = os.path.join(ARTIFACT_ROOT, 'results')
STATS_SAVE_DIR    = os.path.join(ARTIFACT_ROOT, 'stats')

# ── Layer registry -- every sub-layer target Phase 05 trains, its task, and its target column ──
LAYER_TASK = {
    'L1_1_TRI': 'classification', 'L1_1_DIR': 'classification',
    'L1_2_RVOL': 'regression', 'L1_2_MAXRET': 'regression', 'L1_2_MAXDRAW': 'regression',
    'L1_3_RET': 'regression', 'L1_3_PCT': 'regression',
}
LAYER_TARGET_COL = {
    'L1_1_TRI': 'target_tri_h', 'L1_1_DIR': 'target_dir_h',
    'L1_2_RVOL': 'target_fwd_rvol_h', 'L1_2_MAXRET': 'target_max_ret_h', 'L1_2_MAXDRAW': 'target_max_draw_h',
    'L1_3_RET': 'target_fwd_ret_h', 'L1_3_PCT': 'target_fwd_pct_h',
}
ALL_LAYERS = list(LAYER_TASK.keys())
CLASSIFICATION_LAYERS = [l for l, t in LAYER_TASK.items() if t == 'classification']
REGRESSION_LAYERS     = [l for l, t in LAYER_TASK.items() if t == 'regression']

# ── L1.1 (classification) class encoding -- IDENTICAL to Phase05_Unified_L1_v4 ──────────────
CLASS_ORDER      = [-1, 0, 1]
REGIME_LABEL_MAP = {-1: 'Bear', 0: 'Sideways', 1: 'Bull'}
ENCODE_MAP        = {-1: 0, 0: 1, 1: 2}
DECODE_MAP        = {v: k for k, v in ENCODE_MAP.items()}

# ── Feature-Selection Config -- IDENTICAL to Phase05_Unified_L1_v4 Section 2.1 (needed so that
#    Section 6's feature-count sweep reruns the EXACT SAME selection pipeline, just at different
#    top_k, rather than a different one that would confound "feature count" with "selection
#    method") ──────────────────────────────────────────────────────────────────────────────
CORR_THRESHOLD  = 0.75
MISSING_MAX     = 0.30
FS_RANDOM_STATE = RANDOM_STATE
FINAL_SELECTION_BASIS = 'shap'
MI_RANK_WEIGHT, SHAP_RANK_WEIGHT = 0.50, 0.50
SHAP_ENABLED, SHAP_MAX_SAMPLES, SHAP_N_ESTIMATORS = True, 2000, 300
META_COLS = {'date', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'adj_close'}

DOMAIN_GROUPS = {
    'core': ['1. Raw Price & Return', '2. Candle-Body & Microstructure', '3. Momentum Indicators',
              '4. Trend Indicators', '5. Volatility Indicators', '6. Volume Indicators',
              '7. Rolling Statistical', '8. Lag Features', '9. Calendar & Regime'],
    'auxiliary': ['10. External Derived Features', '12. Slope & Acceleration Features',
                  '13. Interaction Features', '14. Advanced Regime, Volatility & Percentile'],
    'excluded': ['0. Cross-Market & Relative Strength', '11. Intraday Features'],
}
GROUP_RATIOS = {'core': 0.80, 'auxiliary': 0.20, 'excluded': 0.00}

# ── Regime-label economics (must match Phase-03's target_tri_{h}d construction) ─────────────
REGIME_THRESHOLD = 0.005

# ── Purged Walk-Forward CV (Section 3) ──────────────────────────────────────────────────────
USE_PURGED_CV = True
PURGE_DAYS    = HORIZON
EMBARGO_DAYS  = HORIZON
WF_SPLITS     = 5   # more folds than Phase 05's stacking OOF (3) -- this notebook's CV is used
                     # for fold-level STABILITY diagnostics (Sections 8 / Q5), which wants more,
                     # smaller folds than an OOF-feature generator does.

# ── This notebook's ONE fixed, default-hyperparameter model, used for every retraining-based
#    check below (Sections 6/8, Q4/Q5) -- same "carefully selected model, not more algorithms"
#    principle as Phase 05's own Section 9b ablation toolkit. ────────────────────────────────
ROBUSTNESS_MODEL_NAME = 'LightGBM_Robustness'
FEATURE_COUNT_GRID    = [20, 40, 60, 80, 100]

for _d in (MODEL_SAVE_DIR, PRED_SAVE_DIR, FEATIMP_SAVE_DIR, RESULTS_SAVE_DIR, STATS_SAVE_DIR):
    os.makedirs(_d, exist_ok=True)

print(f'✅ Config loaded. Default sweep target: SYMBOL={SYMBOL}  HORIZON={HORIZON}d')
print(f'   ALL_SYMBOLS={ALL_SYMBOLS}')
print(f'   ALL_HORIZONS_FOR_STATS={ALL_HORIZONS_FOR_STATS}')
print(f'   ALL_LAYERS={ALL_LAYERS}')
print(f'   ARTIFACT_ROOT = {ARTIFACT_ROOT}')


### 2.1 · Domain / Pattern Rules & Information-Set Buckets — IDENTICAL to Phase 05

`DOMAIN_PATTERN_RULES` (fine-grained, 15 domains) is copied verbatim from Phase 05's Section 2.2
-- used here for the feature-count sweep's own `select_features_train_only()` call and for
`mi_shap_*.csv`'s `domain` column. `INFO_SET_PATTERN_RULES` (coarse, 6 buckets: Market / Macro /
GPR / Sentiment / Sustainability / Regime, plus "Cross-market" added here for the
`0. Cross-Market & Relative Strength` domain the review's Q2 explicitly lists) is copied verbatim
from Phase 05 L2/L3's Section 9b ablation toolkit -- used for Phase 07's domain-level SHAP
aggregation.


In [ ]:
# ── Feature-name -> fine-grained domain (IDENTICAL to Phase05_Unified_L1_v4 Section 2.2) ──────
DOMAIN_PATTERN_RULES = [
    ('peer_', '0. Cross-Market & Relative Strength'), ('xmk_', '0. Cross-Market & Relative Strength'),
    ('rel_mom_', '0. Cross-Market & Relative Strength'), ('rel_atr_', '0. Cross-Market & Relative Strength'),
    ('rel_rsi_', '0. Cross-Market & Relative Strength'), ('rel_drawdown_', '0. Cross-Market & Relative Strength'),
    ('rel_breakout_', '0. Cross-Market & Relative Strength'), ('outperform_', '0. Cross-Market & Relative Strength'),

    ('price_', '1. Raw Price & Return'), ('log_close', '1. Raw Price & Return'),
    ('log_volume', '1. Raw Price & Return'), ('ret_', '1. Raw Price & Return'),
    ('log_ret', '1. Raw Price & Return'), ('gap', '1. Raw Price & Return'),
    ('range_', '1. Raw Price & Return'), ('dist_from_', '1. Raw Price & Return'),
    ('drawdown_', '1. Raw Price & Return'), ('distance_52w', '1. Raw Price & Return'),
    ('illiquidity_', '1. Raw Price & Return'),

    ('candle_', '2. Candle-Body & Microstructure'), ('is_doji', '2. Candle-Body & Microstructure'),
    ('is_hammer', '2. Candle-Body & Microstructure'), ('is_shooting_star', '2. Candle-Body & Microstructure'),
    ('close_in_range', '2. Candle-Body & Microstructure'),

    ('mom_', '3. Momentum Indicators'), ('trend_', '4. Trend Indicators'), ('vola_', '5. Volatility Indicators'),

    ('vol_of_vol_', '14. Advanced Regime, Volatility & Percentile'),
    ('vol_overnight_20', '14. Advanced Regime, Volatility & Percentile'),
    ('vol_intraday_20', '14. Advanced Regime, Volatility & Percentile'),
    ('overnight_intraday_vol_ratio', '14. Advanced Regime, Volatility & Percentile'),
    ('regime_p_', '14. Advanced Regime, Volatility & Percentile'), ('pctile_', '14. Advanced Regime, Volatility & Percentile'),
    ('z_volume_', '14. Advanced Regime, Volatility & Percentile'), ('z_atr_', '14. Advanced Regime, Volatility & Percentile'),
    ('z_mom_', '14. Advanced Regime, Volatility & Percentile'), ('z_vix_', '14. Advanced Regime, Volatility & Percentile'),
    ('days_since_', '14. Advanced Regime, Volatility & Percentile'), ('bull_streak', '14. Advanced Regime, Volatility & Percentile'),
    ('bear_streak', '14. Advanced Regime, Volatility & Percentile'), ('vol_spike_streak', '14. Advanced Regime, Volatility & Percentile'),
    ('consec_', '14. Advanced Regime, Volatility & Percentile'), ('atr_slope', '14. Advanced Regime, Volatility & Percentile'),
    ('atr_accel', '14. Advanced Regime, Volatility & Percentile'), ('atr_increasing', '14. Advanced Regime, Volatility & Percentile'),
    ('realized_quarticity', '14. Advanced Regime, Volatility & Percentile'), ('bipower_variation', '14. Advanced Regime, Volatility & Percentile'),
    ('jump_component', '14. Advanced Regime, Volatility & Percentile'),

    ('vol_', '6. Volume Indicators'), ('stat_', '7. Rolling Statistical'), ('lag_', '8. Lag Features'),
    ('cal_', '9. Calendar & Regime'), ('regime_', '9. Calendar & Regime'),
    ('intra_', '11. Intraday Features'),
    ('india_vix_slope', '12. Slope & Acceleration Features'), ('india_vix_accel', '12. Slope & Acceleration Features'),
    ('gold_trend_slope', '12. Slope & Acceleration Features'), ('yield_spread_slope', '12. Slope & Acceleration Features'),
    ('sentiment_slope', '12. Slope & Acceleration Features'),
    ('ix3_', '13. Interaction Features'), ('ix_', '13. Interaction Features'),

    ('beta_', '10. External Derived Features'), ('corr_', '10. External Derived Features'),
    ('rs_', '10. External Derived Features'), ('gpr_', '10. External Derived Features'),
    ('sustain_gpr', '10. External Derived Features'), ('sent_', '10. External Derived Features'),
    ('net_sentiment', '10. External Derived Features'), ('av_sent', '10. External Derived Features'),
    ('gdelt1_', '10. External Derived Features'), ('gdelt2_', '10. External Derived Features'),
    ('riskcat_', '10. External Derived Features'), ('count_financial', '10. External Derived Features'),
    ('count_political', '10. External Derived Features'), ('count_climate', '10. External Derived Features'),
    ('event_', '10. External Derived Features'), ('high_event', '10. External Derived Features'),
    ('fred_', '10. External Derived Features'), ('rate_', '10. External Derived Features'),
    ('fx_', '10. External Derived Features'), ('inf_', '10. External Derived Features'),
    ('mon_', '10. External Derived Features'), ('act_', '10. External Derived Features'),
    ('lead_', '10. External Derived Features'), ('trade_', '10. External Derived Features'),
    ('yield_curve_inverted', '10. External Derived Features'), ('cpu_', '10. External Derived Features'),
    ('climate_', '10. External Derived Features'), ('green_transition', '10. External Derived Features'),
    ('esg_', '10. External Derived Features'), ('sustain_', '10. External Derived Features'),
    ('india_vix', '10. External Derived Features'), ('usdinr', '10. External Derived Features'),
    ('gold_usd', '10. External Derived Features'), ('gold_safehaven', '10. External Derived Features'),
    ('market_stress', '10. External Derived Features'), ('ext_missing_source', '10. External Derived Features'),
    ('ext_sentiment_momentum', '10. External Derived Features'), ('ext_', '10. External Derived Features'),
]


def get_domain(col: str) -> str:
    for prefix, domain in DOMAIN_PATTERN_RULES:
        if col.startswith(prefix):
            return domain
    return 'Unmapped'


def domain_breakdown(feature_list):
    doms = pd.Series([get_domain(f) for f in feature_list], name='domain')
    out = doms.value_counts().rename_axis('domain').reset_index(name='n_features')
    return out.sort_values('n_features', ascending=False).reset_index(drop=True)


# ── Fine-grained domain -> coarse information-set bucket (IDENTICAL bucketing rule to Phase05
#    L2/L3's Section 9b, PLUS 'Cross-market' for the review's Q2 list) -- built from the SAME
#    prefixes so a feature maps to exactly one bucket regardless of which notebook asks. ─────
INFO_SET_PATTERN_RULES = [
    ('peer_', 'Cross-market'), ('xmk_', 'Cross-market'), ('rel_mom_', 'Cross-market'),
    ('rel_atr_', 'Cross-market'), ('rel_rsi_', 'Cross-market'), ('rel_drawdown_', 'Cross-market'),
    ('rel_breakout_', 'Cross-market'), ('outperform_', 'Cross-market'),
    ('regime_', 'Regime'),
    ('gpr_', 'GPR'), ('sustain_gpr', 'GPR'),
    ('sent_', 'Sentiment'), ('net_sentiment', 'Sentiment'), ('av_sent', 'Sentiment'),
    ('gdelt1_', 'Sentiment'), ('gdelt2_', 'Sentiment'), ('riskcat_', 'Sentiment'),
    ('count_financial', 'Sentiment'), ('count_political', 'Sentiment'), ('count_climate', 'Sentiment'),
    ('event_', 'Sentiment'), ('high_event', 'Sentiment'),
    ('esg_', 'Sustainability'), ('sustain_', 'Sustainability'), ('climate_', 'Sustainability'),
    ('green_transition', 'Sustainability'), ('cpu_', 'Sustainability'),
    ('fred_', 'Macro'), ('rate_', 'Macro'), ('fx_', 'Macro'), ('inf_', 'Macro'), ('mon_', 'Macro'),
    ('act_', 'Macro'), ('lead_', 'Macro'), ('trade_', 'Macro'), ('yield_curve_inverted', 'Macro'),
    ('yield_spread_slope', 'Macro'), ('usdinr', 'Macro'), ('gold_usd', 'Macro'),
    ('gold_trend_slope', 'Macro'), ('gold_safehaven', 'Macro'), ('india_vix', 'Macro'),
    ('market_stress', 'Macro'), ('ext_missing_source', 'Macro'), ('ext_sentiment_momentum', 'Macro'),
    ('ext_', 'Macro'),
]
INFO_SET_CUMULATIVE_ORDER = ['Market', 'Macro', 'GPR', 'Sentiment', 'Sustainability']


def get_info_set(col: str) -> str:
    for prefix, bucket in INFO_SET_PATTERN_RULES:
        if col.startswith(prefix):
            return bucket
    return 'Market'


print(f'✅ DOMAIN_PATTERN_RULES: {len(DOMAIN_PATTERN_RULES)} rules -> '
      f'{len(set(d for _, d in DOMAIN_PATTERN_RULES))} fine-grained domains.')
print(f'✅ INFO_SET_PATTERN_RULES: {len(INFO_SET_PATTERN_RULES)} rules -> coarse buckets '
      f'{sorted(set(b for _, b in INFO_SET_PATTERN_RULES) | {"Market"})}')


## 3 · Load Frozen Phase-03 Features + Frozen Phase-05 Feature-Selection Results

Per the review's outline, this notebook loads the SAME `{symbol}_features_{split}.parquet`
matrices Phase 05 reads (`load_raw_splits`, `clean_xy` — ported verbatim), plus whichever of Phase
05's ALREADY-SAVED, TRAIN-only-selected feature lists it finds under `FS_DIR` (written there by
Phase 05's own `save_selected_features()`). Nothing is re-selected unless a symbol/layer/horizon
combination has genuinely never been run in Phase 05 yet, in which case Section 6's feature-count
sweep runs `select_features_train_only()` itself (also ported verbatim) — the review's own
"feature-count sensitivity" ask is, after all, a request to re-run selection at different `top_k`.


In [ ]:
def load_raw_splits(symbol: str) -> dict:
    """IDENTICAL to Phase05_Unified_L1_v4's load_raw_splits(). Already contains every target_*
    column computed upstream in Phase-03 -- no target is ever recreated here."""
    sym_dir = os.path.join(FEAT_DIR, symbol)
    splits = {}
    for split in ['train', 'val', 'test']:
        path = os.path.join(sym_dir, f'{symbol}_features_{split}.parquet')
        if os.path.exists(path):
            splits[split] = pd.read_parquet(path)
        else:
            print(f'  ⚠️ Not found: {path}')
    return splits


def clean_xy(df: pd.DataFrame, feat_cols: list, target_col: str) -> pd.DataFrame:
    """IDENTICAL to Phase05_Unified_L1_v4's clean_xy()."""
    cols = feat_cols + [target_col]
    d = df[cols].replace([np.inf, -np.inf], np.nan).dropna(subset=[target_col]).copy()
    d[feat_cols] = d[feat_cols].fillna(d[feat_cols].median()).fillna(0)
    return d


def get_candidate_features(df: pd.DataFrame) -> list:
    """IDENTICAL to Phase05_Unified_L1_v4's get_candidate_features()."""
    return [c for c in df.columns if c not in META_COLS and not c.startswith('target_')
            and pd.api.types.is_numeric_dtype(df[c])]


def load_frozen_selection(symbol: str, layer: str, horizon: int):
    """Reads Phase 05's ALREADY-SAVED selection for this (symbol, layer, horizon), written by
    save_selected_features() under FS_DIR/{symbol}/{layer}/{symbol}_selection_{layer}_{h}d.txt
    (+ the matching _ranking_{layer}_{h}d.csv). Returns (feats, ranking_df) or ([], None) if
    Phase 05 has not been run for this combination yet."""
    sel_dir = os.path.join(FS_DIR, symbol, layer)
    feat_path = os.path.join(sel_dir, f'{symbol}_selection_{layer}_{horizon}d.txt')
    rank_path = os.path.join(sel_dir, f'{symbol}_ranking_{layer}_{horizon}d.csv')
    if not os.path.exists(feat_path):
        return [], None
    with open(feat_path) as f:
        feats = [line.strip() for line in f if line.strip()]
    ranking = pd.read_csv(rank_path) if os.path.exists(rank_path) else None
    return feats, ranking


def build_xy_from_frozen(symbol, layer, horizon, split, raw_splits=None):
    """Loads `split` for `symbol` (or reuses `raw_splits` if already loaded) and slices it down
    to Phase 05's frozen selected features for (symbol, layer, horizon). Returns (X, y, index) or
    None if either the split or the frozen selection is unavailable."""
    raw_splits = raw_splits if raw_splits is not None else load_raw_splits(symbol)
    df = raw_splits.get(split)
    feats, _ = load_frozen_selection(symbol, layer, horizon)
    target_col = LAYER_TARGET_COL[layer]
    if df is None or not feats or target_col not in df.columns:
        return None
    d = clean_xy(df, feats, target_col)
    return d[feats], d[target_col], d.index


print('load_raw_splits(), clean_xy(), get_candidate_features(), load_frozen_selection(), '
      'build_xy_from_frozen() ready.')


### 3.1 · Train-Only Selection Pipeline (ported verbatim) — used only by Section 6's feature-count sweep


In [ ]:
def _rank_to_score(rank_series):
    n = rank_series.notna().sum()
    if n <= 1:
        return pd.Series(1.0, index=rank_series.index, dtype=float)
    return 1.0 - (rank_series - 1) / (n - 1)


def _compute_shap_importance(X_imp, y, features, task, random_state):
    """IDENTICAL to Phase05_Unified_L1_v4's _compute_shap_importance(): train-only SHAP via an RF
    surrogate, on up to SHAP_MAX_SAMPLES rows. Reused by both the feature-count sweep (Section 6)
    and every SHAP computation in Phase 07."""
    if not SHAP_ENABLED or not features:
        return pd.Series(dtype=float, name='shap_importance')
    try:
        X_sel = X_imp[features].copy()
        max_samples = min(SHAP_MAX_SAMPLES, len(X_sel))
        if max_samples <= 0:
            return pd.Series(dtype=float, name='shap_importance')
        X_shap = X_sel.sample(n=max_samples, random_state=random_state) if len(X_sel) > max_samples else X_sel
        y_shap = y.loc[X_shap.index]
        if task == 'classification':
            surrogate = RandomForestClassifier(n_estimators=SHAP_N_ESTIMATORS, random_state=random_state,
                                                n_jobs=-1, class_weight='balanced')
        else:
            surrogate = RandomForestRegressor(n_estimators=SHAP_N_ESTIMATORS, random_state=random_state, n_jobs=-1)
        surrogate.fit(X_shap, y_shap)
        explainer = shap.TreeExplainer(surrogate)
        shap_values = explainer.shap_values(X_shap)
        if isinstance(shap_values, list):
            arr = np.stack([np.asarray(v) for v in shap_values], axis=0)
            mean_abs = np.mean(np.abs(arr), axis=(0, 1))
        else:
            arr = np.asarray(shap_values)
            mean_abs = np.mean(np.abs(arr), axis=0) if arr.ndim == 2 else np.mean(np.abs(arr), axis=(0, 2))
        return pd.Series(mean_abs, index=features, name='shap_importance').sort_values(ascending=False)
    except Exception as exc:
        print(f'  ⚠️ SHAP importance failed; returning empty. Reason: {type(exc).__name__}: {exc}')
        return pd.Series(dtype=float, name='shap_importance')


def select_features_train_only(train_df, candidate_features, target_col, task='classification',
                                top_k=40, corr_threshold=CORR_THRESHOLD, missing_max=MISSING_MAX,
                                random_state=FS_RANDOM_STATE, domain_groups=DOMAIN_GROUPS,
                                group_ratios=GROUP_RATIOS, verbose=True):
    """IDENTICAL pipeline to Phase05_Unified_L1_v4's select_features_train_only(): missingness ->
    variance -> correlation -> MI -> SHAP -> Unmapped removal -> domain-group cap -> final top_k.
    The ONLY reason this notebook ever calls it is Section 6's feature-count sweep -- everywhere
    else uses Phase 05's already-frozen selection (load_frozen_selection above)."""
    X = train_df[candidate_features].apply(pd.to_numeric, errors='coerce')
    y = train_df[target_col]
    valid = y.notna()
    X, y = X[valid], y[valid]

    missing_pct = X.isna().mean()
    keep = missing_pct[missing_pct <= missing_max].index.tolist()
    X = X[keep]

    X_imp_all = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=X.columns, index=X.index)
    std = X_imp_all.std()
    variance_keep = std[std > 1e-12].index.tolist()
    X_imp = X_imp_all[variance_keep]
    if X_imp.shape[1] == 0:
        return [], pd.DataFrame(columns=['feature', 'mi', 'mi_rank', 'shap_importance', 'shap_rank', 'domain', 'group'])

    mi_fn = mutual_info_classif if task == 'classification' else mutual_info_regression
    y_mi = y.astype(int) if task == 'classification' else y.astype(float)
    mi_pre = mi_fn(X_imp, y_mi, random_state=random_state)
    pre_order = pd.Series(mi_pre, index=X_imp.columns).sort_values(ascending=False).index.tolist()

    keep_corr = []
    corr = X_imp[pre_order].corr().abs()
    for feature in pre_order:
        if not keep_corr or float(corr.loc[feature, keep_corr].max()) < corr_threshold:
            keep_corr.append(feature)
    X_corr = X_imp[keep_corr]

    mi_final = mi_fn(X_corr, y_mi, random_state=random_state)
    ranking = pd.DataFrame({'feature': X_corr.columns, 'mi': mi_final}).sort_values('mi', ascending=False).reset_index(drop=True)
    ranking['mi_rank'] = ranking.index + 1
    ranking['domain'] = ranking['feature'].map(get_domain)

    shap_imp = _compute_shap_importance(X_corr, y, ranking['feature'].tolist(), task, random_state)
    ranking['shap_importance'] = ranking['feature'].map(shap_imp)
    ranking['shap_rank'] = ranking['feature'].map({f: i + 1 for i, f in enumerate(shap_imp.index)}) if len(shap_imp) else np.nan

    mapped_mask = ranking['domain'].ne('Unmapped')
    ranking_mapped = ranking.loc[mapped_mask].copy()
    if ranking_mapped.empty:
        return [], ranking

    domain_groups = domain_groups or {}
    group_ratios = group_ratios or {}
    domain_to_group = {d: g for g, doms in domain_groups.items() for d in doms}
    ranking_mapped['group'] = ranking_mapped['domain'].map(lambda d: domain_to_group.get(d, 'core'))
    ranking_mapped['mi_score'] = _rank_to_score(ranking_mapped['mi_rank'])
    ranking_mapped['shap_score'] = _rank_to_score(ranking_mapped['shap_rank'])

    if FINAL_SELECTION_BASIS == 'mi':
        ranking_mapped['final_score'] = ranking_mapped['mi_score']
        ranking_mapped = ranking_mapped.sort_values(['final_score', 'mi', 'feature'], ascending=[False, False, True])
    elif FINAL_SELECTION_BASIS == 'shap':
        ranking_mapped['final_score'] = ranking_mapped['shap_score']
        ranking_mapped = ranking_mapped.sort_values(['final_score', 'shap_importance', 'feature'], ascending=[False, False, True], na_position='last')
    else:
        w_mi = MI_RANK_WEIGHT / (MI_RANK_WEIGHT + SHAP_RANK_WEIGHT)
        w_shap = SHAP_RANK_WEIGHT / (MI_RANK_WEIGHT + SHAP_RANK_WEIGHT)
        ranking_mapped['final_score'] = w_mi * ranking_mapped['mi_score'] + w_shap * ranking_mapped['shap_score'].fillna(0.0)
        ranking_mapped = ranking_mapped.sort_values(['final_score', 'mi', 'shap_importance', 'feature'], ascending=[False, False, False, True], na_position='last')

    excluded_groups = {g for g, r in group_ratios.items() if r == 0}
    group_caps = {g: round(top_k * r) for g, r in group_ratios.items() if g not in excluded_groups}
    eligible = ranking_mapped[~ranking_mapped['group'].isin(excluded_groups)].copy()

    selected, group_counts = [], {}
    for _, row in eligible.iterrows():
        if len(selected) >= top_k:
            break
        g = row['group']
        cap = group_caps.get(g)
        if cap is not None and group_counts.get(g, 0) >= cap:
            continue
        selected.append(row['feature'])
        group_counts[g] = group_counts.get(g, 0) + 1
    if len(selected) < min(top_k, len(eligible)):
        for _, row in eligible.iterrows():
            if len(selected) >= top_k:
                break
            if row['feature'] not in selected:
                selected.append(row['feature'])

    if verbose:
        print(f'  select_features_train_only: top_k={top_k} -> selected {len(selected)} features '
              f'from {len(eligible)} group-eligible candidates')
    return selected, ranking


print('_compute_shap_importance(), select_features_train_only() ready (ported verbatim from Phase 05).')


## 4 · Purged Walk-Forward CV (embargo) — IDENTICAL to Phase05_Unified_L1_v4 Section 7b

Used by Section 8 (Walk-Forward Stability) and Q5 (SHAP fold-stability) below to generate
leakage-safe, non-overlapping-at-the-boundary folds for the fixed `ROBUSTNESS_MODEL_NAME` model.


In [ ]:
class PurgedWalkForwardCV:
    """IDENTICAL to Phase05_Unified_L1_v4's PurgedWalkForwardCV -- expanding-window CV with a
    purge window before, and an embargo window after, each validation fold."""
    def __init__(self, n_splits=5, purge=0, embargo=0):
        self.n_splits = n_splits
        self.purge = max(int(purge), 0)
        self.embargo = max(int(embargo), 0)

    def split(self, X, y=None, groups=None):
        n = len(X)
        tscv = TimeSeriesSplit(n_splits=self.n_splits)
        for train_idx, val_idx in tscv.split(X):
            val_start, val_end = val_idx[0], val_idx[-1]
            purge_start = max(val_start - self.purge, 0)
            train_idx_purged = train_idx[train_idx < purge_start]
            embargo_end = min(val_end + self.embargo + 1, n)
            train_idx_final = train_idx_purged[(train_idx_purged < val_start) | (train_idx_purged >= embargo_end)]
            if len(train_idx_final) == 0 or len(val_idx) == 0:
                continue
            yield train_idx_final, val_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


def get_cv_splitter(n_splits=WF_SPLITS):
    if USE_PURGED_CV:
        return PurgedWalkForwardCV(n_splits=n_splits, purge=PURGE_DAYS, embargo=EMBARGO_DAYS)
    return TimeSeriesSplit(n_splits=n_splits)


print(f'PurgedWalkForwardCV / get_cv_splitter() ready. USE_PURGED_CV={USE_PURGED_CV} '
      f'n_splits={WF_SPLITS} purge={PURGE_DAYS}d embargo={EMBARGO_DAYS}d')


## 5 · Shared Evaluation Helpers + the ONE Fixed Robustness Model

`evaluate_classification()` / `evaluate_regression()` are ported verbatim from Phase 05 (so every
metric name and definition matches exactly). `build_lightgbm_fixed()` is the single,
default-hyperparameter model every retraining-based section below uses -- deliberately not tuned
per arm, so that any difference across feature counts / folds / regimes reflects the *condition*
being swept, not a hyperparameter search finding a different optimum each time.


In [ ]:
def evaluate_classification(y_true, y_pred, y_proba=None):
    y_true = pd.Series(y_true).astype(int).values
    y_pred = pd.Series(y_pred).astype(int).values
    row = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', labels=CLASS_ORDER, zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', labels=CLASS_ORDER, zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', labels=CLASS_ORDER, zero_division=0),
        'cohen_kappa': cohen_kappa_score(y_true, y_pred, labels=CLASS_ORDER),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'roc_auc_ovr': np.nan, 'log_loss': np.nan,
    }
    if y_proba is not None:
        try:
            y_true_bin = label_binarize(y_true, classes=CLASS_ORDER)
            row['roc_auc_ovr'] = roc_auc_score(y_true_bin, y_proba, average='macro', multi_class='ovr')
        except Exception:
            pass
        try:
            row['log_loss'] = log_loss(y_true, y_proba, labels=CLASS_ORDER)
        except Exception:
            pass
    return row


def evaluate_regression(y_true, y_pred, y_ref=None, dir_mode='median', n_features=None):
    y_true = np.asarray(y_true, dtype=float); y_pred = np.asarray(y_pred, dtype=float)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {'rmse': np.nan, 'mae': np.nan, 'r2': np.nan, 'spearman_corr': np.nan,
                'directional_acc': np.nan, 'forecast_bias': np.nan}
    r2 = r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan
    row = {'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))), 'mae': float(mean_absolute_error(y_true, y_pred)),
           'r2': float(r2), 'spearman_corr': np.nan, 'directional_acc': np.nan,
           'forecast_bias': float(np.mean(y_pred - y_true))}
    try:
        corr, _ = spearmanr(y_true, y_pred)
        row['spearman_corr'] = float(corr)
    except Exception:
        pass
    if dir_mode == 'zero':
        row['directional_acc'] = float(np.mean(np.sign(y_true) == np.sign(y_pred)))
    else:
        ref = y_ref if y_ref is not None else np.median(y_true)
        row['directional_acc'] = float(((y_true >= ref).astype(int) == (y_pred >= ref).astype(int)).mean())
    return row


def evaluate(task, y_true, y_pred, y_proba=None, y_ref=None, dir_mode='median', n_features=None):
    return (evaluate_classification(y_true, y_pred, y_proba=y_proba) if task == 'classification'
            else evaluate_regression(y_true, y_pred, y_ref=y_ref, dir_mode=dir_mode, n_features=n_features))


def build_lightgbm_fixed(task):
    """The ONE model every Section-6/8/Q4/Q5 retraining call below uses -- default
    hyperparameters, identical shape to Phase05's own build_lightgbm()."""
    default = dict(n_estimators=300, num_leaves=31, learning_rate=0.05, subsample=0.8,
                    colsample_bytree=0.8, random_state=RANDOM_STATE, verbosity=-1)
    if task == 'classification':
        default['objective'] = 'multiclass' if len(CLASS_ORDER) > 2 else 'binary'
        if len(CLASS_ORDER) > 2:
            default['num_class'] = len(CLASS_ORDER)
        return lgb.LGBMClassifier(**default)
    default['objective'] = 'regression'
    return lgb.LGBMRegressor(**default)


print('evaluate(), build_lightgbm_fixed(task) ready.')


## 6 · Cross-Grid Artifact Loader

One function, `load_artifact_grid(subdir, filename)`, powers almost every aggregation section
below: it scans `Artifacts/{subdir}/{symbol}/{layer}/h{horizon}d/{filename}` across every
`(symbol, layer, horizon)` in `ALL_SYMBOLS × ALL_LAYERS × ALL_HORIZONS_FOR_STATS`, reads whatever
Phase 05 has already saved there, and concatenates it into one long-format DataFrame — tagged with
the `symbol`/`layer`/`horizon` it came from even if the file itself didn't already have those
columns. Missing combinations (a symbol/horizon/layer Phase 05 hasn't been run for yet) are
silently skipped, exactly like Phase 05's own `aggregate_*_across_horizons()` helpers, just
generalised to the full grid instead of one symbol/layer at a time.


In [ ]:
def load_artifact_grid(subdir, filename, symbols=ALL_SYMBOLS, layers=None,
                        horizons=ALL_HORIZONS_FOR_STATS, verbose=False):
    """subdir: one of 'results', 'stats', 'feature_importance', 'predictions'. filename: e.g.
    'leaderboard.csv', 'information_set_ablation.csv', 'diebold_mariano.csv', 'mi_shap_train.csv'.
    layers=None -> every layer in ALL_LAYERS. Returns one concatenated DataFrame (empty if
    nothing found), with symbol/layer/horizon columns added whenever the source file lacks them
    (most already carry these -- see Phase05's save functions -- but this makes the loader safe
    regardless)."""
    layers = layers or ALL_LAYERS
    root = {'results': RESULTS_SAVE_DIR, 'stats': STATS_SAVE_DIR,
            'feature_importance': FEATIMP_SAVE_DIR, 'predictions': PRED_SAVE_DIR}[subdir]
    frames = []
    n_found, n_missing = 0, 0
    for symbol in symbols:
        for layer in layers:
            for horizon in horizons:
                path = os.path.join(root, symbol, layer, f'h{horizon}d', filename)
                if not os.path.exists(path):
                    n_missing += 1
                    continue
                try:
                    df = pd.read_csv(path)
                except (pd.errors.EmptyDataError, pd.errors.ParserError):
                    n_missing += 1
                    continue
                if df.empty:
                    continue
                for col, val in [('symbol', symbol), ('layer', layer), ('horizon', horizon)]:
                    if col not in df.columns:
                        df.insert(0, col, val)
                frames.append(df)
                n_found += 1
    if verbose or n_found == 0:
        print(f'  load_artifact_grid({subdir}/*/{{layer}}/h*/{filename}): '
              f'{n_found} files found, {n_missing} combinations not yet run in Phase 05.')
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


print('load_artifact_grid(subdir, filename, symbols=..., layers=..., horizons=...) ready.')


## Q1 · Which Features Matter? — Feature-Level SHAP, Aggregated

Reads every `mi_shap_train.csv` / `mi_shap_test.csv` Phase 05 has already saved (written by
`save_train_side_importance_copy()` / `compute_test_side_importance()` in each of L1/L2/L3),
across the full `ALL_SYMBOLS × ALL_LAYERS × ALL_HORIZONS_FOR_STATS` grid. No retraining — this is
exactly the TRAIN-only SHAP ranking each model was actually selected on.


In [ ]:
shap_train_grid = load_artifact_grid('feature_importance', 'mi_shap_train.csv', verbose=True)
shap_test_grid  = load_artifact_grid('feature_importance', 'mi_shap_test.csv', verbose=True)
print(f'\nTRAIN-side SHAP rows: {len(shap_train_grid)}  |  TEST-side SHAP rows: {len(shap_test_grid)}')

if not shap_train_grid.empty:
    top_features_by_layer = (
        shap_train_grid.groupby(['layer', 'feature'])['shap_importance']
        .mean().reset_index()
        .sort_values(['layer', 'shap_importance'], ascending=[True, False])
    )
    print('\nTop-10 features by mean TRAIN-side SHAP importance, per layer (averaged across every '
          'symbol/horizon already run):')
    for layer, g in top_features_by_layer.groupby('layer'):
        print(f'\n{layer}:')
        display(g.head(10).reset_index(drop=True))

    out_dir = os.path.join(FEATIMP_SAVE_DIR, '_phase07_summary')
    os.makedirs(out_dir, exist_ok=True)
    shap_train_grid.to_csv(os.path.join(out_dir, 'mi_shap_train_full_grid.csv'), index=False)
    if not shap_test_grid.empty:
        shap_test_grid.to_csv(os.path.join(out_dir, 'mi_shap_test_full_grid.csv'), index=False)
    top_features_by_layer.to_csv(os.path.join(out_dir, 'top_features_by_layer.csv'), index=False)
    print(f'\n💾 Saved -> {out_dir}')
else:
    print('  ⚠️ Nothing found yet -- run Phase05 for at least one symbol/layer/horizon first.')


## Q2 · Which Information Domains Matter? — SHAP Aggregated by Bucket

Buckets every feature in the TRAIN-side SHAP grid into `Market / Macro / GPR / Sentiment /
Sustainability / Cross-market / Regime` (Section 2.1's `get_info_set()` — the SAME bucketing rule
Phase 05 L2/L3's H2 ablation uses), sums `shap_importance` within each bucket per
(symbol, layer, horizon), then reports each bucket's SHARE of total SHAP mass — the direct answer
to "which information domains matter", comparable across layers.


In [ ]:
if not shap_train_grid.empty:
    shap_train_grid['info_set'] = shap_train_grid['feature'].map(get_info_set)
    domain_shap = (
        shap_train_grid.groupby(['symbol', 'layer', 'horizon', 'info_set'])['shap_importance']
        .sum().reset_index()
    )
    domain_shap['share_of_total'] = domain_shap.groupby(['symbol', 'layer', 'horizon'])['shap_importance'].transform(
        lambda s: s / s.sum() if s.sum() > 0 else np.nan)

    domain_share_by_layer = (
        domain_shap.groupby(['layer', 'info_set'])['share_of_total']
        .mean().reset_index()
        .sort_values(['layer', 'share_of_total'], ascending=[True, False])
    )
    print('Mean SHAP share of total, by information-set bucket, per layer '
          '(averaged across every symbol/horizon already run):')
    for layer, g in domain_share_by_layer.groupby('layer'):
        print(f'\n{layer}:')
        display(g.reset_index(drop=True))

    fig, axes = plt.subplots(1, len(ALL_LAYERS), figsize=(4 * len(ALL_LAYERS), 4), sharey=True)
    for ax, layer in zip(np.atleast_1d(axes), ALL_LAYERS):
        g = domain_share_by_layer[domain_share_by_layer['layer'] == layer]
        if g.empty:
            ax.set_visible(False)
            continue
        ax.barh(g['info_set'], g['share_of_total'])
        ax.set_title(layer, fontsize=9)
        ax.invert_yaxis()
    plt.suptitle('Mean SHAP share of total by information-set bucket')
    plt.tight_layout()
    plt.show()

    out_dir = os.path.join(FEATIMP_SAVE_DIR, '_phase07_summary')
    os.makedirs(out_dir, exist_ok=True)
    domain_shap.to_csv(os.path.join(out_dir, 'domain_shap_full_grid.csv'), index=False)
    domain_share_by_layer.to_csv(os.path.join(out_dir, 'domain_shap_share_by_layer.csv'), index=False)
    print(f'\n💾 Saved -> {out_dir}')
else:
    print('  ⚠️ Nothing to bucket yet -- run Q1\'s cell above first (needs Phase 05 artifacts).')


## Q3 · Does Importance Change With Horizon? — Domain Shares Across h=1,3,5,10,21

Same domain-bucket SHAP shares as Q2, now sliced by `horizon` instead of averaged over it — turns
the report's own "external-derived features become more prominent at longer horizons" observation
into an actual cross-horizon table (and chart) instead of a narrative claim, directly supporting
**H4**.


In [ ]:
if not shap_train_grid.empty:
    domain_share_by_horizon = (
        domain_shap.groupby(['layer', 'horizon', 'info_set'])['share_of_total']
        .mean().reset_index()
    )
    for layer in ALL_LAYERS:
        g = domain_share_by_horizon[domain_share_by_horizon['layer'] == layer]
        if g.empty:
            continue
        pivot = g.pivot_table(index='info_set', columns='horizon', values='share_of_total')
        print(f'\n{layer} — mean SHAP share by information-set bucket, across horizon:')
        display(pivot)

        external_buckets = [b for b in ['Macro', 'GPR', 'Sentiment', 'Sustainability'] if b in pivot.index]
        if external_buckets:
            external_share = pivot.loc[external_buckets].sum()
            print(f'  Combined EXTERNAL (Macro+GPR+Sentiment+Sustainability) share by horizon: '
                  + ', '.join(f'h={h}d: {v:.1%}' for h, v in external_share.items()))

    out_dir = os.path.join(FEATIMP_SAVE_DIR, '_phase07_summary')
    os.makedirs(out_dir, exist_ok=True)
    domain_share_by_horizon.to_csv(os.path.join(out_dir, 'domain_shap_share_by_horizon.csv'), index=False)
    print(f'\n💾 Saved -> {out_dir}')
else:
    print('  ⚠️ Nothing to slice yet -- run Q1/Q2\'s cells above first.')


## Q4 · Does Importance Change With Regime? — SHAP Recomputed per Bull/Bear/Sideways (new)

Phase 05 never computes SHAP conditional on market regime, so this section retrains: for the
current `SYMBOL`/`HORIZON` (Section 2) and every layer, loads the frozen selected features
(Section 3), joins in Phase 05 L1's `target_tri_{h}d` regime label for the SAME TRAIN+VAL rows,
fits the **same fixed** `build_lightgbm_fixed()` model **separately on each regime's rows**, and
computes SHAP (`_compute_shap_importance()`, Section 3.1) within each. Reports each feature's
mean `|SHAP|` per regime, plus the resulting information-set-bucket shares per regime — directly
supporting **H5**.

Caveat, stated plainly: fitting a SEPARATE model per regime bucket (rather than one model with a
regime-interaction term) means a feature's SHAP importance here reflects how it drives the target
*within* that regime specifically, on however many rows that regime has in TRAIN+VAL — regimes
with few rows (typically Sideways is the largest, Bear the smallest) give noisier SHAP estimates.


In [ ]:
def shap_by_regime(symbol, horizon, layer, min_rows=100):
    task = LAYER_TASK[layer]
    res_tr = build_xy_from_frozen(symbol, layer, horizon, 'train')
    res_va = build_xy_from_frozen(symbol, layer, horizon, 'val')
    if res_tr is None or res_va is None:
        print(f'  ⚠️ {layer}: no frozen selection / data for {symbol} h={horizon}d -- skipping.')
        return pd.DataFrame()
    X_tr, y_tr, idx_tr = res_tr
    X_va, y_va, idx_va = res_va
    X = pd.concat([X_tr, X_va]).sort_index()
    y = pd.concat([y_tr, y_va]).sort_index()

    raw_splits = load_raw_splits(symbol)
    regime_frames = [df[['target_tri_h']] for df in (raw_splits.get('train'), raw_splits.get('val'))
                      if df is not None and 'target_tri_h' in df.columns]
    if not regime_frames:
        print(f'  ⚠️ {symbol}: target_tri_h not found in train/val -- cannot bucket by regime, skipping {layer}.')
        return pd.DataFrame()
    regime_labels = pd.concat(regime_frames)['target_tri_h'].reindex(X.index).map(REGIME_LABEL_MAP)

    rows = []
    for regime_name in ['Bull', 'Sideways', 'Bear']:
        mask = regime_labels == regime_name
        X_r, y_r = X[mask], y[mask]
        if len(X_r) < min_rows:
            print(f'  ⚠️ {layer} | {regime_name}: only {len(X_r)} rows (< min_rows={min_rows}) -- skipping.')
            continue
        shap_imp = _compute_shap_importance(X_r, y_r, list(X_r.columns), task, RANDOM_STATE)
        for feat, val in shap_imp.items():
            rows.append({'symbol': symbol, 'layer': layer, 'horizon': horizon, 'regime': regime_name,
                          'n_rows': len(X_r), 'feature': feat, 'shap_importance': val,
                          'domain': get_domain(feat), 'info_set': get_info_set(feat)})
        print(f'  {layer} | {regime_name}: {len(X_r)} rows, top feature = '
              f'{shap_imp.index[0] if len(shap_imp) else "n/a"}')

    df = pd.DataFrame(rows)
    if not df.empty:
        out_dir = os.path.join(FEATIMP_SAVE_DIR, symbol, layer, f'h{horizon}d')
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, 'shap_by_regime.csv')
        df.to_csv(out_path, index=False)
        print(f'  💾 Saved -> {out_path}')
    return df


shap_regime_results = {}
for _layer in ALL_LAYERS:
    print(f'\n{"=" * 90}\nSHAP by regime | {SYMBOL} | h={HORIZON}d | {_layer}\n{"=" * 90}')
    shap_regime_results[_layer] = shap_by_regime(SYMBOL, HORIZON, _layer)

shap_regime_this_run = pd.concat([df for df in shap_regime_results.values() if not df.empty], ignore_index=True) \
    if any(not df.empty for df in shap_regime_results.values()) else pd.DataFrame()
if not shap_regime_this_run.empty:
    domain_share_regime = (
        shap_regime_this_run.groupby(['layer', 'regime', 'info_set'])['shap_importance']
        .sum().reset_index()
    )
    domain_share_regime['share_of_total'] = domain_share_regime.groupby(['layer', 'regime'])['shap_importance'].transform(
        lambda s: s / s.sum() if s.sum() > 0 else np.nan)
    for layer in ALL_LAYERS:
        g = domain_share_regime[domain_share_regime['layer'] == layer]
        if g.empty:
            continue
        pivot = g.pivot_table(index='info_set', columns='regime', values='share_of_total')
        print(f'\n{layer} — SHAP share by information-set bucket, across regime:')
        display(pivot)


### Q4b · Aggregate SHAP-by-Regime across ALL symbols/horizons already run


In [ ]:
shap_by_regime_grid = load_artifact_grid('feature_importance', 'shap_by_regime.csv', verbose=True)
print(f'\nTotal SHAP-by-regime rows across the full grid: {len(shap_by_regime_grid)}')
if not shap_by_regime_grid.empty:
    domain_share_regime_all = (
        shap_by_regime_grid.groupby(['layer', 'regime', 'info_set'])['shap_importance']
        .sum().reset_index()
    )
    domain_share_regime_all['share_of_total'] = domain_share_regime_all.groupby(['layer', 'regime'])['shap_importance'].transform(
        lambda s: s / s.sum() if s.sum() > 0 else np.nan)
    print('Mean SHAP share by information-set bucket, across regime, across every symbol/horizon already run:')
    display(domain_share_regime_all)

    out_dir = os.path.join(FEATIMP_SAVE_DIR, '_phase07_summary')
    os.makedirs(out_dir, exist_ok=True)
    shap_by_regime_grid.to_csv(os.path.join(out_dir, 'shap_by_regime_full_grid.csv'), index=False)
    domain_share_regime_all.to_csv(os.path.join(out_dir, 'domain_shap_share_by_regime.csv'), index=False)
    print(f'\n💾 Saved -> {out_dir}')


## Q5 · Is Importance Stable? — SHAP Recomputed per Purged Walk-Forward Fold (new)

For the current `SYMBOL`/`HORIZON` and every layer, runs `PurgedWalkForwardCV` (Section 4) over
TRAIN+VAL's frozen selected features, fits the **same fixed** `build_lightgbm_fixed()` model on
each fold's training slice, and computes SHAP on that fold's own held-out validation slice. A
feature's **rank-stability** is the Spearman correlation of its importance rank between
consecutive folds, averaged across all fold-pairs; a feature's **magnitude-stability** is
`1 - (std / mean)` of its `|SHAP|` across folds (clipped at 0). Both are reported per feature and
rolled up to a single, layer-level stability score.


In [ ]:
def shap_fold_stability(symbol, horizon, layer):
    task = LAYER_TASK[layer]
    res_tr = build_xy_from_frozen(symbol, layer, horizon, 'train')
    res_va = build_xy_from_frozen(symbol, layer, horizon, 'val')
    if res_tr is None or res_va is None:
        print(f'  ⚠️ {layer}: no frozen selection / data for {symbol} h={horizon}d -- skipping.')
        return pd.DataFrame(), pd.DataFrame()
    X_tr, y_tr, _ = res_tr
    X_va, y_va, _ = res_va
    X = pd.concat([X_tr, X_va]).sort_index()
    y = pd.concat([y_tr, y_va]).sort_index()

    cv = get_cv_splitter(WF_SPLITS)
    fold_shap = {}
    for fold_i, (train_idx, val_idx) in enumerate(cv.split(X)):
        X_fold_tr, y_fold_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_fold_va, y_fold_va = X.iloc[val_idx], y.iloc[val_idx]
        if len(X_fold_va) < 20:
            continue
        shap_imp = _compute_shap_importance(X_fold_va, y_fold_va, list(X_fold_va.columns), task, RANDOM_STATE)
        fold_shap[fold_i] = shap_imp

    if len(fold_shap) < 2:
        print(f'  ⚠️ {layer}: fewer than 2 usable folds -- cannot assess stability, skipping.')
        return pd.DataFrame(), pd.DataFrame()

    shap_by_fold_df = pd.DataFrame(fold_shap)  # rows=feature, columns=fold
    shap_by_fold_df.index.name = 'feature'

    # Rank-stability: mean pairwise Spearman correlation of feature ranks between consecutive folds.
    fold_ids = sorted(fold_shap.keys())
    rank_corrs = []
    for a, b in zip(fold_ids[:-1], fold_ids[1:]):
        common = shap_by_fold_df[[a, b]].dropna()
        if len(common) < 3:
            continue
        corr, _ = spearmanr(common[a], common[b])
        rank_corrs.append(corr)
    mean_rank_stability = float(np.nanmean(rank_corrs)) if rank_corrs else np.nan

    # Per-feature magnitude-stability: 1 - (std/mean) of |SHAP| across folds, clipped at 0.
    feat_mean = shap_by_fold_df.mean(axis=1)
    feat_std = shap_by_fold_df.std(axis=1)
    magnitude_stability = (1 - (feat_std / feat_mean.replace(0, np.nan))).clip(lower=0)

    feature_stability_df = pd.DataFrame({
        'symbol': symbol, 'layer': layer, 'horizon': horizon, 'feature': shap_by_fold_df.index,
        'mean_shap': feat_mean.values, 'std_shap': feat_std.values,
        'magnitude_stability': magnitude_stability.values, 'domain': [get_domain(f) for f in shap_by_fold_df.index],
        'info_set': [get_info_set(f) for f in shap_by_fold_df.index],
    }).sort_values('mean_shap', ascending=False)

    layer_summary_row = {'symbol': symbol, 'layer': layer, 'horizon': horizon, 'n_folds': len(fold_shap),
                          'mean_rank_stability_spearman': mean_rank_stability,
                          'mean_magnitude_stability': float(magnitude_stability.mean())}

    print(f'  {layer}: {len(fold_shap)} folds | mean rank-stability (Spearman)={mean_rank_stability:.3f} | '
          f'mean magnitude-stability={layer_summary_row["mean_magnitude_stability"]:.3f}')

    out_dir = os.path.join(FEATIMP_SAVE_DIR, symbol, layer, f'h{horizon}d')
    os.makedirs(out_dir, exist_ok=True)
    feature_stability_df.to_csv(os.path.join(out_dir, 'shap_fold_stability_features.csv'), index=False)
    pd.DataFrame([layer_summary_row]).to_csv(os.path.join(out_dir, 'shap_fold_stability_summary.csv'), index=False)
    print(f'  💾 Saved -> {out_dir}')
    return feature_stability_df, pd.DataFrame([layer_summary_row])


shap_stability_feature_results, shap_stability_summary_results = {}, []
for _layer in ALL_LAYERS:
    print(f'\n{"=" * 90}\nSHAP fold-stability | {SYMBOL} | h={HORIZON}d | {_layer}\n{"=" * 90}')
    feat_df, summary_df = shap_fold_stability(SYMBOL, HORIZON, _layer)
    shap_stability_feature_results[_layer] = feat_df
    if not summary_df.empty:
        shap_stability_summary_results.append(summary_df)

if shap_stability_summary_results:
    display(pd.concat(shap_stability_summary_results, ignore_index=True))


### Q5b · Aggregate SHAP Fold-Stability across ALL symbols/horizons already run


In [ ]:
shap_stability_summary_grid = load_artifact_grid('feature_importance', 'shap_fold_stability_summary.csv', verbose=True)
shap_stability_feature_grid = load_artifact_grid('feature_importance', 'shap_fold_stability_features.csv', verbose=True)
print(f'\nTotal SHAP fold-stability summary rows: {len(shap_stability_summary_grid)}  |  '
      f'feature-level rows: {len(shap_stability_feature_grid)}')

if not shap_stability_summary_grid.empty:
    print('Mean rank/magnitude stability by layer, across every symbol/horizon already run:')
    display(shap_stability_summary_grid.groupby('layer')[['mean_rank_stability_spearman', 'mean_magnitude_stability']]
            .mean().reset_index().sort_values('mean_rank_stability_spearman'))

if not shap_stability_feature_grid.empty:
    least_stable = (
        shap_stability_feature_grid.sort_values('magnitude_stability')
        .groupby('layer').head(5)[['layer', 'feature', 'domain', 'mean_shap', 'magnitude_stability']]
    )
    print('\n5 LEAST stable high-importance features per layer (lowest magnitude_stability):')
    display(least_stable)

out_dir = os.path.join(FEATIMP_SAVE_DIR, '_phase07_summary')
os.makedirs(out_dir, exist_ok=True)
if not shap_stability_summary_grid.empty:
    shap_stability_summary_grid.to_csv(os.path.join(out_dir, 'shap_fold_stability_summary_full_grid.csv'), index=False)
if not shap_stability_feature_grid.empty:
    shap_stability_feature_grid.to_csv(os.path.join(out_dir, 'shap_fold_stability_features_full_grid.csv'), index=False)
print(f'\n💾 Saved -> {out_dir}')


## Final · Combined XAI & Information-Dynamics Summary

One consolidated table per layer: its single top feature and top information-set bucket (Q1/Q2),
whether the external-information share rises with horizon (Q3/H4), which regime its importance
profile differs most under (Q4/H5), and its mean SHAP fold-stability (Q5) — the table meant to
anchor the report's explainability section.


In [ ]:
xai_summary_rows = []
for layer in ALL_LAYERS:
    row = {'layer': layer}

    if not shap_train_grid.empty:
        g = top_features_by_layer[top_features_by_layer['layer'] == layer]
        row['top_feature'] = g.iloc[0]['feature'] if not g.empty else None
    else:
        row['top_feature'] = None

    if 'domain_share_by_layer' in dir() and not domain_share_by_layer.empty:
        g = domain_share_by_layer[domain_share_by_layer['layer'] == layer]
        row['top_info_set_bucket'] = g.iloc[0]['info_set'] if not g.empty else None
    else:
        row['top_info_set_bucket'] = None

    if 'domain_share_by_horizon' in dir() and not domain_share_by_horizon.empty:
        g = domain_share_by_horizon[(domain_share_by_horizon['layer'] == layer)
                                     & (domain_share_by_horizon['info_set'].isin(['Macro', 'GPR', 'Sentiment', 'Sustainability']))]
        if not g.empty:
            ext_by_h = g.groupby('horizon')['share_of_total'].sum().sort_index()
            row['external_share_rises_with_horizon'] = bool(ext_by_h.iloc[-1] > ext_by_h.iloc[0]) if len(ext_by_h) > 1 else None
        else:
            row['external_share_rises_with_horizon'] = None
    else:
        row['external_share_rises_with_horizon'] = None

    if not shap_by_regime_grid.empty:
        g = shap_by_regime_grid[shap_by_regime_grid['layer'] == layer]
        if not g.empty:
            regime_totals = g.groupby('regime')['shap_importance'].sum()
            row['regime_with_highest_total_shap'] = regime_totals.idxmax() if not regime_totals.empty else None
        else:
            row['regime_with_highest_total_shap'] = None
    else:
        row['regime_with_highest_total_shap'] = None

    if not shap_stability_summary_grid.empty:
        g = shap_stability_summary_grid[shap_stability_summary_grid['layer'] == layer]
        row['mean_rank_stability'] = float(g['mean_rank_stability_spearman'].mean()) if not g.empty else np.nan
    else:
        row['mean_rank_stability'] = np.nan

    xai_summary_rows.append(row)

xai_summary = pd.DataFrame(xai_summary_rows)
print('Combined XAI & Information-Dynamics Summary — one row per layer:')
display(xai_summary)

out_dir = os.path.join(FEATIMP_SAVE_DIR, '_phase07_summary')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'xai_information_dynamics_summary.csv')
xai_summary.to_csv(out_path, index=False)
print(f'\n💾 Saved final XAI summary -> {out_path}')
